# La prise en charge d’Unicode par le langage Python

Python 3 introduit une révolution dans la gestion de l’encodage des caractères, au prix d’une confusion troublante pour qui vient de Python 2 :

- en Python 2, les chaînes de caractères (type `str`) sont par défaut encodées
- en Python 3, elles sont décodées

Type par défaut d’une chaîne de caractères décodée en Python 3 :

In [ ]:
# decoded string
type('chaîne de caractères')

Une chaîne encodée **ne** doit contenir **que** des caractères ASCII :

In [ ]:
# encoded string (type 'bytes')
print(b'chaîne de caractères')

Pour la manipuler sous forme d’octets, il faut alors l’encoder en choisissant un encodage :

In [ ]:
print(
    'chaîne de caractères'.encode('utf8'),
    bytes('chaîne de caractères', 'latin1'),
    sep="\n"
)

À l’inverse, une liste d’octets en entrée doit être décodée pour s’afficher :

In [ ]:
segment = 'chaîne de caractères'.encode('utf8')

print(segment.decode('utf8'))

Il convient donc de connaître l’encodage utilisé, sinon le décodage se passe mal :

In [ ]:
print(
    segment.decode('latin1'),
    segment.decode('mac-roman'),
    segment.decode('cp855'),
    sep="\n"
)

En bref, lorsque Python 3 manipule du texte, il offre toujours un affichage Unicode. Les ennuis commencent lorsque l’on veut manipuler des données binaires :

In [ ]:
a = 'chaîne de caractères'
b = 'chaîne de caractères'.encode('latin1')

print(a[3], b[3], sep="\n")

L’accès atomique à des données binaires se fait donc sous forme d’octet. Pour savoir quel caractère Unicode se cache derrière un entier, on utilise la fonction `chr()` avec un nombre dans un intervalle $[0, 1114111]$ :

In [ ]:
print(
    chr(b[3]),
    chr(238),
    sep="\n"
)

Et pour connaître le point de code Unicode d’un caractère décodé, on utilise la fonction `ord()` :

In [ ]:
letters = ['ε', 'ܬ', '𐄍']

[ ord(l) for l in letters ]

## Cas pratique

En japonais, le mot *paix* s’orthographie *平和* (*heiwa*). Il est constitué de deux caractères :

- *平*
- *和*

Leurs numéros d’ordre Unicode s’obtiennent facilement :

In [ ]:
[ ord(c) for c in '平和' ]

Ces numéros sont fournis en base 10 (décimale). Or, le langage informatique repose sur une base 2 (binaire). Si le premier idéogramme peut se représenter sur 5 signes allant de 0 à 9 (24179), il est impossible de le représenter sur un octet (8 signes allant de 0 à 1).

Comme il faut plus d’un octet pour le représenter, toute conversion est impossible :

In [ ]:
print(b'平')

Pour rendre la conversion possible, on doit d’abord fournir un encodage adéquat :

In [ ]:
# 平 is not part of latin1 character set
c = '平'.encode('latin1')

In [ ]:
# UTF-8 works well!
c = '平'.encode('utf8')

**Attention !** `c` est désormais une donnée binaire, représentée sous forme d’octets !

In [ ]:
# how many bytes to encode ideogram 平 in UTF-8?
print(len(c))

# 3 bytes: e5 b9 b3 (in hexa form)
print(c)

# bytes in decimal form
for byte in c:
    print(byte)

# in binary form
for byte in c:
    print(bin(byte))

# f-String with format spectifier
for byte in c:
    print(f"{byte:b}")

## Encodages disponibles avec Python

Le module encodings permet de lister tous les encodages pris en charge par Python :

In [ ]:
import encodings

list_encodings = set(encodings.aliases.aliases.values())

sorted_list = sorted(list_encodings)

print(sorted_list)

## Unicode et Python

Rappelons tout d’abord que toute chaîne affichée par Python3 est compatible Unicode. On peut afficher un caractère en l’imprimant directement :

In [ ]:
print("ᛢ")

Comme en indiquant son point de code Unicode ou son identifiant normalisé :

In [ ]:
print(
    "\u16E2", # four-digit Unicode escape
    "\U0001F334", # eight-digit Unicode escape
    "\xEE", # two-digit hexadecimal escape
    "\N{AVESTAN LETTER SHYE}", # Unicode name
    "\N{LATIN SMALL LETTER A}\N{COMBINING GRAVE ACCENT}", # combining two characters
    sep="\n"
)

Le module qui permet d’interagir avec Unicode s’appelle *unicodedata*. Il expose des méthodes pour manipuler les caractères :

- `.lookup()` : rechercher un caractère par son nom
- `.normalize()` : obtenir une version normalisée du caractère
- `.name()` : obtenir l’identifiant unique du caractère
- …

In [ ]:
import unicodedata as ud

char = '㈎'

print(
    ud.name(char),
    ud.category(char),
    ud.decomposition(char),
    sep="\n"
)

### Ordre lexicographique

Une opération commune en analyse textuelle consiste à trier des éléments (*i.e.* des mots) par ordre alphabétique :

In [ ]:
words = ['lettre', 'apostrophe']
words.sort()

print(words)

Le premier écueil se manifeste dès que des majuscules s’invitent dans la liste :

In [ ]:
words = ['lettre', 'apostrophe', 'Ellipse']
words.sort()

print(words)

C’est ici qu’intervient le paramètre `key` pour transmettre une fonction à chaque élément de la liste, avant toute comparaison :

In [ ]:
words = ['lettre', 'apostrophe', 'Ellipse']

# each word is converted to lowercase before sorting
words.sort(key=str.lower)

print(words)

Méthode plutôt efficace et suffisante pour l’anglais, mais qui montre ses limites dès qu’entrent en jeu des caractères accentués :

In [ ]:
words = ['lettre', 'apostrophe', 'Ellipse', 'à']
words.sort(key=str.lower)

print(words)

L’un des remèdes possibles consiste à supprimer les accents des chaînes de caractères.

Plutôt que de prévoir tous les cas possibles (*Si le caractère se trouve parmi \[à, â, ä\], alors le transformer en "a", et sinon, si le caractère est un parmi \[éè\]…*), la méthode `.normalize()` du module `unicodedata` permet d’effectuer ce calcul plus simplement.

En résumé, la fonction décompose un caractère en ses différents composants. Exemple avec la "ligature minuscule latine fi" :

In [ ]:
fi = ud.lookup('Latin Small Ligature Fi')
components = ud.normalize('NFKD', fi)

print(f'Le caractère "{ fi }" est composé de : { [ c for c in components ] }')

Appliquée à notre collection de mots, les caractères accentués ne sont plus relégués à la fin :

In [ ]:
words = ['lettre', 'apostrophe', 'Ellipse', 'à']

words.sort(
    key=lambda ch:ud.normalize('NFKD', ch)
)

print(words)

En revanche, elle soulève d’autres problèmes :

- perte de la distinction majuscules/minuscules
- perte du tri par nombre de caractères

Plutôt que de transmettre une fonction anonyme au paramètre `key`, autant définir une fonction utilisateur :

In [ ]:
def comparison(word):
    """Removes the accent marks on every character in a word.
    
    Keyword argument:
    word -- the word to analyze
    """
    norm = ""
    word = word.lower()
    for ch in word:
        components = ud.normalize('NFKD', ch)
        norm += components[0]
    return norm

words = ['lettre', 'apostrophe', 'Ellipse', 'à']
words.sort(key=comparison)

print(words)